In [2]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Sample documents
documents = [
    "This is a list which containing sample documents.",
    "Keywords are important for keyword-based search.",
    "Document analysis involves extracting keywords.",
    "Keyword-based search relies on sparse embeddings."
]

In [4]:
query = "keyword-based search"

In [5]:
import re
def preprocess_text(text):
    # Convert text to lowercase
    text = text.lower()
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    return text

In [6]:
preprocessed_docs = [preprocess_text(doc) for doc in documents]
preprocessed_docs

['this is a list which containig sample documents',
 'keywords are important for keywordbased search',
 'document analysis involves extracting keywords',
 'keywordbased search relies on sparse embeddings']

In [7]:
preprocessed_query = preprocess_text(query)
preprocessed_query

'keywordbased search'

In [8]:
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(preprocessed_docs)

In [9]:
X.toarray()

array([[0.        , 0.        , 0.37796447, 0.        , 0.37796447,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.37796447, 0.        , 0.        , 0.37796447, 0.        ,
        0.        , 0.37796447, 0.        , 0.        , 0.37796447,
        0.37796447],
       [0.        , 0.4533864 , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.4533864 , 0.4533864 , 0.        ,
        0.        , 0.35745504, 0.35745504, 0.        , 0.        ,
        0.        , 0.        , 0.35745504, 0.        , 0.        ,
        0.        ],
       [0.46516193, 0.        , 0.        , 0.46516193, 0.        ,
        0.        , 0.46516193, 0.        , 0.        , 0.46516193,
        0.        , 0.        , 0.36673901, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.43671931, 0.        , 0.        , 0.       

In [10]:
query_embeddings = tfidf.transform([preprocessed_query])
query_embeddings.toarray()

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.70710678, 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.70710678, 0.        , 0.        ,
        0.        ]])

In [11]:
similarities = cosine_similarity(X, query_embeddings)
similarities

array([[0.        ],
       [0.50551777],
       [0.        ],
       [0.48693426]])

In [12]:
# Ranking - Sort similarity scores by index, reverse to descending order, and convert to a flat list of ranked indices.
ranked_indices = np.argsort(similarities,axis=0)[::-1].flatten()
ranked_indices

array([1, 3, 2, 0])

In [13]:
ranked_documents = [documents[i] for i in ranked_indices]

In [14]:
# Output the ranked documents
for i, doc in enumerate(ranked_documents):
    print(f"Rank {i+1}: {doc}")

Rank 1: Keywords are important for keyword-based search.
Rank 2: Keyword-based search relies on sparse embeddings.
Rank 3: Document analysis involves extracting keywords.
Rank 4: This is a list which containig sample documents.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(r'papers/RAG-for-NLP_paper.pdf')
docs = loader.load()

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.split_documents(docs)

In [ ]:
from langchain_cohere import CohereEmbeddings

embeddings = CohereEmbeddings(model="embed-v4.0")

In [18]:
from langchain_community.vectorstores import Chroma

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [19]:
vector_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})
vector_retriever

VectorStoreRetriever(tags=['Chroma', 'CohereEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x79e03e165670>, search_kwargs={'k': 3})

In [20]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

keyword_retriever = BM25Retriever.from_documents(documents=chunks)
keyword_retriever.k = 3

In [21]:
# Mixing vector search and keyword search for Hybrid search
# hybrid_score = (1 — alpha) * sparse_score + alpha * dense_score (below alpha=0.3)
# Internally this formula is not used by Langchain, it uses Reciprocal Rank Fusion(RRF)
ensemble_retriever = EnsembleRetriever(retrievers=[vector_retriever, keyword_retriever], weights=[0.3, 0.7])
ensemble_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['Chroma', 'CohereEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x79e03e165670>, search_kwargs={'k': 3}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x79e0340bc200>, k=3)], weights=[0.3, 0.7])

In [22]:
model_name = "HuggingFaceH4/zephyr-7b-beta"

In [37]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline

In [28]:
def load_quantized_model(model_name: str):
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.bfloat16,
        device_map="auto"
    )
    return model

In [29]:
def initialize_tokenizer(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Only set pad token if missing (common issue with LLaMA-family)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    return tokenizer

In [30]:
tokenizer = initialize_tokenizer(model_name)

In [31]:
model = load_quantized_model(model_name)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [58]:
from transformers import TextStreamer

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    use_cache=True,
    device_map="auto",
    max_new_tokens=256,
    streamer=streamer,
    do_sample=True,
    top_k=5,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,  # use eos as pad for Mistral-family
)

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'pad_token_id', 'eos_token_id', 'use_cache', 'top_k', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [55]:
llm = HuggingFacePipeline(pipeline=generation_pipeline)

In [44]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [45]:
prompt = PromptTemplate.from_template(
    """
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """
)

In [48]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [59]:
parallel_chain = RunnableParallel({
    'context': RunnableSequence(ensemble_retriever, RunnableLambda(format_docs)),
    'question': RunnablePassthrough()
})

main_chain = parallel_chain | prompt | llm | StrOutputParser()

In [ ]:
print(main_chain.invoke("What is Abstractive Question Answering?"))